In [ ]:
# In[1]:

import pandas as pd
from collections import OrderedDict

# Load files
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetime (Unix seconds -> UTC)
for df in (metric_df, trace_df, log_df, error_df):
    if 'timestamp' in df.columns:
        df['ts'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

def compact_timespan(df):
    if df.shape[0] == 0 or 'ts' not in df.columns:
        return None, None
    mn = df['ts'].min()
    mx = df['ts'].max()
    return (mn.isoformat(), mx.isoformat())

def top_counts(series, limit=50):
    if series is None or series.shape[0] == 0:
        return {}
    vc = series.value_counts().head(limit)
    # return as ordered dict for compact JSON-like representation
    return OrderedDict(vc.to_dict())

# Prepare summaries
metric_summary = OrderedDict()
metric_summary['rows'] = int(metric_df.shape[0])
metric_summary['min_ts'], metric_summary['max_ts'] = compact_timespan(metric_df)
metric_summary['cmdb_ids'] = list(metric_df['cmdb_id'].dropna().unique())[:50]
metric_summary['kpi_counts_top_50'] = top_counts(metric_df['kpi_name'], limit=50)

trace_summary = OrderedDict()
trace_summary['rows'] = int(trace_df.shape[0])
trace_summary['min_ts'], trace_summary['max_ts'] = compact_timespan(trace_df)
trace_summary['cmdb_ids'] = list(trace_df['cmdb_id'].dropna().unique())[:50]
trace_summary['feature_counts_top_50'] = top_counts(trace_df['trace_name'], limit=50)

log_summary = OrderedDict()
log_summary['rows'] = int(log_df.shape[0])
log_summary['min_ts'], log_summary['max_ts'] = compact_timespan(log_df)
log_summary['cmdb_ids'] = list(log_df['cmdb_id'].dropna().unique())[:50]
log_summary['feature_counts_top_50'] = top_counts(log_df['log_name'], limit=50)

error_summary = OrderedDict()
error_summary['rows'] = int(error_df.shape[0])
error_summary['min_ts'], error_summary['max_ts'] = compact_timespan(error_df)
error_summary['cmdb_ids'] = list(error_df['cmdb_id'].dropna().unique())[:50]
# message existence and top 5 distinct samples
if 'message' in error_df.columns:
    non_null_messages = error_df['message'].dropna().astype(str)
    error_summary['message_non_null_count'] = int(non_null_messages.shape[0])
    error_summary['top_5_distinct_messages'] = list(pd.Series(non_null_messages.unique()).head(5).tolist())
else:
    error_summary['message_non_null_count'] = 0
    error_summary['top_5_distinct_messages'] = []

# Aggregate into final compact JSON-like dict
summary = OrderedDict([
    ('metric.csv', metric_summary),
    ('trace.csv', trace_summary),
    ('log.csv', log_summary),
    ('error_logs.csv', error_summary)
])

# Display the compact summary (variable output)
summary

```
Out[1]:
```
```python
summary = (
    "metric.csv: 1875 rows; time range 2024-01-19T12:03:00+00:00 to 2024-01-19T12:27:00+00:00. "
    "cmdb_ids (sample): [adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, frontend-external, paymentservice, productcatalogservice, recommendationservice, redis, shippingservice]. "
    "Top KPIs (counts): cpu 275, workload 275, socket 275, mem 275, latency-50 250, latency-90 250, diskio 225, error 50.\n\n"
    "trace.csv: 3800 rows; time range 2024-01-19T12:03:00+00:00 to 2024-01-19T12:27:00+00:00. "
    "cmdb_ids (sample): [checkoutservice, currencyservice, emailservice, frontendservice, paymentservice, productcatalogservice, recommendationservice, root]. "
    "Top trace features: trace.from_root.duration_mean/p95/error_rate/row_count (each 175); many 'from_' and 'to_' service metrics for checkoutservice and frontendservice (125 each) and to_productcatalogservice metrics (100).\n\n"
    "log.csv: 458 rows; time range 2024-01-19T12:03:00+00:00 to 2024-01-19T12:27:00+00:00. "
    "cmdb_ids (sample): [adservice, cartservice, checkoutservice, currencyservice, emailservice, frontend, paymentservice, recommendationservice, shippingservice, redis]. "
    "Top log features: log.error_count 229, log.row_count 229.\n\n"
    "error_logs.csv: 0 rows; no timestamps or cmdb_ids present; no messages recorded."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

OrderedDict([('metric.csv', OrderedDict([('rows', 1875), ('min_ts', '2024-01-19T12:03:00+00:00'), ('max_ts', '2024-01-19T12:27:00+00:00'), ('cmdb_ids', ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'frontend-external', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'redis', 'shippingservice']), ('kpi_counts_top_50', OrderedDict([('cpu', 275), ('workload', 275), ('socket', 275), ('mem', 275), ('latency-50', 250), ('latency-90', 250), ('diskio', 225), ('error', 50)]))])), ('trace.csv', OrderedDict([('rows', 3800), ('min_ts', '2024-01-19T12:03:00+00:00'), ('max_ts', '2024-01-19T12:27:00+00:00'), ('cmdb_ids', ['checkoutservice', 'currencyservice', 'emailservice', 'frontendservice', 'paymentservice', 'productcatalogservice', 'recommendationservice', 'root']), ('feature_counts_top_50', OrderedDict([('trace.from_root.duration_mean', 175), ('trace.from_root.duration_p95', 175), ('trace.from_root.error_rate', 175), ('trace.from_root.row_count', 175), ('trace.from_checkoutservice.row_count', 125), ('trace.from_checkoutservice.error_rate', 125), ('trace.from_checkoutservice.duration_p95', 125), ('trace.from_checkoutservice.duration_mean', 125), ('trace.from_frontendservice.row_count', 125), ('trace.from_frontendservice.error_rate', 125), ('trace.from_frontendservice.duration_p95', 125), ('trace.from_frontendservice.duration_mean', 125), ('trace.to_productcatalogservice.row_count', 100), ('trace.to_productcatalogservice.error_rate', 100), ('trace.to_productcatalogservice.duration_p95', 100), ('trace.to_productcatalogservice.duration_mean', 100), ('trace.to_checkoutservice.duration_mean', 75), ('trace.to_checkoutservice.duration_p95', 75), ('trace.to_checkoutservice.error_rate', 75), ('trace.to_checkoutservice.row_count', 75), ('trace.to_recommendationservice.duration_mean', 75), ('trace.to_recommendationservice.duration_p95', 75), ('trace.to_recommendationservice.error_rate', 75), ('trace.to_recommendationservice.row_count', 75), ('trace.to_currencyservice.duration_mean', 75), ('trace.to_currencyservice.duration_p95', 75), ('trace.to_currencyservice.error_rate', 75), ('trace.to_currencyservice.row_count', 75), ('trace.to_paymentservice.row_count', 50), ('trace.to_paymentservice.error_rate', 50), ('trace.to_paymentservice.duration_p95', 50), ('trace.to_paymentservice.duration_mean', 50), ('trace.to_emailservice.row_count', 50), ('trace.to_emailservice.error_rate', 50), ('trace.to_emailservice.duration_p95', 50), ('trace.to_emailservice.duration_mean', 50), ('trace.to_frontendservice.row_count', 50), ('trace.to_frontendservice.error_rate', 50), ('trace.to_frontendservice.duration_p95', 50), ('trace.to_frontendservice.duration_mean', 50), ('trace.from_recommendationservice.duration_mean', 50), ('trace.from_recommendationservice.duration_p95', 50), ('trace.from_recommendationservice.error_rate', 50), ('trace.from_recommendationservice.row_count', 50)]))])), ('log.csv', OrderedDict([('rows', 458), ('min_ts', '2024-01-19T12:03:00+00:00'), ('max_ts', '2024-01-19T12:27:00+00:00'), ('cmdb_ids', ['adservice', 'cartservice', 'checkoutservice', 'currencyservice', 'emailservice', 'frontend', 'paymentservice', 'recommendationservice', 'shippingservice', 'redis']), ('feature_counts_top_50', OrderedDict([('log.error_count', 229), ('log.row_count', 229)]))])), ('error_logs.csv', OrderedDict([('rows', 0), ('min_ts', None), ('max_ts', None), ('cmdb_ids', []), ('message_non_null_count', 0), ('top_5_distinct_messages', [])]))])```
```

In [ ]:
# In[2]:

import pandas as pd
from collections import OrderedDict

# reuse previously loaded dataframes if available; otherwise load
try:
    metric_df
except NameError:
    metric_df = pd.read_csv("metric.csv")
try:
    log_df
except NameError:
    log_df = pd.read_csv("log.csv")

# ensure ts columns (UTC) exist
if 'ts' not in metric_df.columns:
    metric_df['ts'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
if 'ts' not in log_df.columns:
    log_df['ts'] = pd.to_datetime(log_df['timestamp'], unit='s', utc=True)

# Candidate services
services = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Incident window (UTC)
window_start = pd.to_datetime("2024-01-19 12:00:11", utc=True)
window_end = pd.to_datetime("2024-01-19 12:30:11", utc=True)

# -------- METRIC ANALYSIS --------
# 1) Global P95 per (cmdb_id, kpi_name) using entire metric_df
metric_p95 = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# 2) Filter metric rows to incident window
metric_window = metric_df[(metric_df['ts'] >= window_start) & (metric_df['ts'] <= window_end)].copy()

# Keep only candidate services
metric_window = metric_window[metric_window['cmdb_id'].isin(services)]

# Merge global P95 into windowed rows
metric_merged = metric_window.merge(metric_p95, on=['cmdb_id', 'kpi_name'], how='left')

# Aggregate per series
def agg_metric_group(df):
    count_in_window = int(df.shape[0])
    above_mask = df['value'] > df['global_P95']
    count_above = int(above_mask.sum())
    if count_above > 0:
        earliest_above = df.loc[above_mask, 'ts'].min().isoformat()
    else:
        earliest_above = None
    max_in_window = float(df['value'].max())
    # global_P95 is same for group
    gp95 = float(df['global_P95'].iloc[0]) if pd.notna(df['global_P95'].iloc[0]) else None
    return pd.Series({
        'global_P95': gp95,
        'count_points_in_window': count_in_window,
        'count_points_above_P95': count_above,
        'earliest_timestamp_of_point_above_P95': earliest_above,
        'max_value_in_window': max_in_window
    })

metric_results = (
    metric_merged
    .groupby(['cmdb_id', 'kpi_name'], dropna=False)
    .apply(agg_metric_group)
    .reset_index()
)

# Only include series that have at least 1 row in the window (groupby from metric_merged ensures that)
# Sort by count_points_above_P95 desc and limit top 20
metric_results_sorted = metric_results.sort_values(by='count_points_above_P95', ascending=False).head(20)

# If no anomalies across all returned rows, prepare a note
metric_anomalies_exist = (metric_results_sorted['count_points_above_P95'].sum() > 0)
metric_note = None
if not metric_anomalies_exist:
    metric_note = "No anomalies found: count_points_above_P95 == 0 for all series in the window."

# -------- LOG ANALYSIS --------
# Focus only on log.error_count and log.row_count for same services
log_features = ['log.error_count', 'log.row_count']

# 1) Global P95 per (cmdb_id, log_name) using entire log_df
log_p95 = (
    log_df[log_df['log_name'].isin(log_features)]
    .groupby(['cmdb_id', 'log_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# 2) Filter log rows to incident window and candidate services and features
log_window = log_df[
    (log_df['ts'] >= window_start) &
    (log_df['ts'] <= window_end) &
    (log_df['cmdb_id'].isin(services)) &
    (log_df['log_name'].isin(log_features))
].copy()

# Merge global P95 into windowed log rows
log_merged = log_window.merge(log_p95, on=['cmdb_id', 'log_name'], how='left')

# Aggregate per series
def agg_log_group(df):
    count_in_window = int(df.shape[0])
    above_mask = df['value'] > df['global_P95']
    count_above = int(above_mask.sum())
    if count_above > 0:
        earliest_above = df.loc[above_mask, 'ts'].min().isoformat()
    else:
        earliest_above = None
    max_in_window = float(df['value'].max())
    gp95 = float(df['global_P95'].iloc[0]) if pd.notna(df['global_P95'].iloc[0]) else None
    return pd.Series({
        'global_P95': gp95,
        'count_points_in_window': count_in_window,
        'count_points_above_P95': count_above,
        'earliest_timestamp_of_point_above_P95': earliest_above,
        'max_value_in_window': max_in_window
    })

if log_merged.shape[0] > 0:
    log_results = (
        log_merged
        .groupby(['cmdb_id', 'log_name'], dropna=False)
        .apply(agg_log_group)
        .reset_index()
    )
    log_results_sorted = log_results.sort_values(by='count_points_above_P95', ascending=False).head(20)
else:
    # empty DataFrame with expected columns
    log_results_sorted = pd.DataFrame(columns=['cmdb_id','log_name','global_P95','count_points_in_window',
                                              'count_points_above_P95','earliest_timestamp_of_point_above_P95',
                                              'max_value_in_window'])

log_anomalies_exist = (log_results_sorted['count_points_above_P95'].sum() if not log_results_sorted.empty else 0) > 0
log_note = None
if not log_anomalies_exist:
    log_note = "No anomalies found: count_points_above_P95 == 0 for all log.series in the window."

# Keep final outputs compact: selected columns and top rows
metric_out = metric_results_sorted[['cmdb_id','kpi_name','global_P95','count_points_in_window',
                                   'count_points_above_P95','earliest_timestamp_of_point_above_P95',
                                   'max_value_in_window']].reset_index(drop=True)

log_out = log_results_sorted[['cmdb_id','log_name','global_P95','count_points_in_window',
                              'count_points_above_P95','earliest_timestamp_of_point_above_P95',
                              'max_value_in_window']].reset_index(drop=True)

# Display compact results (variable output)
metric_out, metric_note, log_out, log_note

```
Out[2]:
```
metric_out = metric_out  # from previous analysis
log_out = log_out

# Summarize metric anomalies by service
metric_anom = metric_out[metric_out['count_points_above_P95'] > 0].copy()
metric_by_service = {}
for svc, g in metric_anom.groupby('cmdb_id'):
    items = []
    for _, row in g.iterrows():
        items.append(
            f"{row['kpi_name']} ({int(row['count_points_above_P95'])} pts, earliest {row['earliest_timestamp_of_point_above_P95']}, max {row['max_value_in_window']:.6g})"
        )
    metric_by_service[svc] = "; ".join(items)

# Summarize log anomalies by service
log_anom = log_out.copy()
log_rowcount = log_anom[log_anom['log_name'] == 'log.row_count']
log_errorcount = log_anom[log_anom['log_name'] == 'log.error_count']

log_by_service = {}
for svc, g in log_rowcount.groupby('cmdb_id'):
    row = g.iloc[0]
    log_by_service.setdefault(svc, []).append(
        f"log.row_count ({int(row['count_points_above_P95'])} pts, earliest {row['earliest_timestamp_of_point_above_P95']}, max {row['max_value_in_window']:.6g})"
    )
for svc, g in log_errorcount.groupby('cmdb_id'):
    row = g.iloc[0]
    log_by_service.setdefault(svc, []).append(
        f"log.error_count ({int(row['count_points_above_P95'])} pts, earliest {row['earliest_timestamp_of_point_above_P95']}, max {row['max_value_in_window']:.6g})"
    )

# Build plain-English summary
lines = []
lines.append("Summary of anomalies in the incident window 2024-01-19 12:00:11 UTC -> 2024-01-19 12:30:11 UTC:")
lines.append("")
# Metrics
if metric_anom.empty:
    lines.append("Metrics: No anomalies found (count_points_above_P95 == 0 for all series).")
else:
    total_metric_series = metric_anom.shape[0]
    total_metric_anom_points = int(metric_anom['count_points_above_P95'].sum())
    lines.append(f"Metrics: {total_metric_series} series had values > global P95, totaling {total_metric_anom_points} point(s) above P95.")
    for svc in sorted(metric_by_service.keys()):
        lines.append(f"- {svc}: {metric_by_service[svc]}")
lines.append("")

# Logs
if log_out.empty:
    lines.append("Logs: No relevant log series in the window for the selected services.")
else:
    # Check error_count anomalies
    total_log_anom_points = int(log_out['count_points_above_P95'].sum())
    if total_log_anom_points == 0:
        lines.append("Logs: No anomalies found for log.row_count or log.error_count (count_points_above_P95 == 0).")
    else:
        lines.append(f"Logs: total {total_log_anom_points} point(s) above global P95 across the checked log.series.")
        for svc in sorted(log_by_service.keys()):
            lines.append(f"- {svc}: {'; '.join(log_by_service[svc])}")
        # Note if error_count had no anomalies
        if log_errorcount['count_points_above_P95'].sum() == 0:
            lines.append("Note: log.error_count showed no points above P95 for these services (all zero).")

summary = "\n".join(lines)
summary

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id    kpi_name    global_P95  count_points_in_window  count_points_above_P95 earliest_timestamp_of_point_above_P95  max_value_in_window
0         checkoutservice         cpu  5.348422e-01                    25.0                     2.0             2024-01-19T12:17:00+00:00         6.077457e-01
1         checkoutservice  latency-50  2.259881e+00                    25.0                     2.0             2024-01-19T12:19:00+00:00         2.397604e+00
2         checkoutservice  latency-90  4.375236e+00                    25.0                     2.0             2024-01-19T12:19:00+00:00         4.449817e+00
3         checkoutservice         mem  1.213604e+07                    25.0                     2.0             2024-01-19T12:21:00+00:00         1.218970e+07
4         checkoutservice    workload  1.197820e+00                    25.0                     2.0             2024-01-19T12:04:00+00:00         1.211150e+00
5         currencyservice         cpu  3.372133e+01                    25.0                     2.0             2024-01-19T12:16:00+00:00         3.401223e+01
6         currencyservice  latency-90  5.957678e-02                    25.0                     2.0             2024-01-19T12:24:00+00:00         6.402294e-02
7         currencyservice  latency-50  4.237573e-03                    25.0                     2.0             2024-01-19T12:03:00+00:00         4.370273e-03
8         currencyservice    workload  8.235938e+01                    25.0                     2.0             2024-01-19T12:07:00+00:00         8.470872e+01
9         currencyservice         mem  4.563055e+07                    25.0                     2.0             2024-01-19T12:13:00+00:00         4.583929e+07
10           emailservice         mem  4.351473e+07                    25.0                     2.0             2024-01-19T12:24:00+00:00         4.352949e+07
11           emailservice  latency-90  2.203132e-01                    25.0                     2.0             2024-01-19T12:22:00+00:00         2.221105e-01
12           emailservice         cpu  3.392264e-01                    25.0                     2.0             2024-01-19T12:06:00+00:00         3.399208e-01
13           emailservice  latency-50  6.667590e-02                    25.0                     2.0             2024-01-19T12:22:00+00:00         1.026359e-01
14  productcatalogservice  latency-50  1.994296e-03                    25.0                     2.0             2024-01-19T12:17:00+00:00         1.998830e-03
15  productcatalogservice  latency-90  4.453280e-03                    25.0                     2.0             2024-01-19T12:18:00+00:00         4.457809e-03
16  productcatalogservice         cpu  3.886458e+00                    25.0                     2.0             2024-01-19T12:17:00+00:00         3.953080e+00
17           emailservice    workload  1.189527e+00                    25.0                     2.0             2024-01-19T12:06:00+00:00         1.225683e+00
18  recommendationservice    workload  1.957094e+01                    25.0                     2.0             2024-01-19T12:06:00+00:00         1.966993e+01
19  recommendationservice         mem  4.521166e+07                    25.0                     2.0             2024-01-19T12:17:00+00:00         4.522175e+07, None,                  cmdb_id         log_name  global_P95  count_points_in_window  count_points_above_P95 earliest_timestamp_of_point_above_P95  max_value_in_window
0        checkoutservice    log.row_count       107.8                    25.0                     2.0             2024-01-19T12:04:00+00:00                117.0
1        currencyservice    log.row_count      2465.6                    25.0                     2.0             2024-01-19T12:07:00+00:00               2515.0
2  recommendationservice    log.row_count       589.4                    25.0                     2.0             2024-01-19T12:06:00+00:00                598.0
3           emailservice    log.row_count        36.0                    25.0                     1.0             2024-01-19T12:06:00+00:00                 39.0
4        currencyservice  log.error_count         0.0                    25.0                     0.0                                   NaN                  0.0
5        checkoutservice  log.error_count         0.0                    25.0                     0.0                                   NaN                  0.0
6           emailservice  log.error_count         0.0                    25.0                     0.0                                   NaN                  0.0
7  recommendationservice  log.error_count         0.0                    25.0                     0.0                                   NaN                  0.0, None)```
```

In [ ]:
# In[3]:

import pandas as pd
from collections import OrderedDict

# Reuse existing variables if present, otherwise load files
try:
    trace_df
except NameError:
    trace_df = pd.read_csv("trace.csv")

# Ensure ts parsed to UTC datetime
if 'ts' not in trace_df.columns:
    trace_df['ts'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)

# Candidate services
try:
    services
except NameError:
    services = ['checkoutservice', 'currencyservice', 'emailservice', 'productcatalogservice', 'recommendationservice']

# Incident window (UTC)
try:
    window_start, window_end
except NameError:
    window_start = pd.to_datetime("2024-01-19 12:00:11", utc=True)
    window_end = pd.to_datetime("2024-01-19 12:30:11", utc=True)

# 1) Compute global P95 per (cmdb_id, trace_name) across full trace_df
trace_p95 = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], dropna=False)['value']
    .quantile(0.95)
    .reset_index()
    .rename(columns={'value': 'global_P95'})
)

# 2) Filter trace rows to incident window
trace_window = trace_df[(trace_df['ts'] >= window_start) & (trace_df['ts'] <= window_end)].copy()

# 3) Select rows where cmdb_id is in services OR trace_name contains any service name
# Build regex for contains (escape service names if needed)
service_regex = "|".join(services)
mask = trace_window['cmdb_id'].isin(services) | trace_window['trace_name'].str.contains(service_regex, na=False)
trace_selected = trace_window[mask].copy()

# Merge global P95 into the selected windowed rows
trace_merged = trace_selected.merge(trace_p95, on=['cmdb_id', 'trace_name'], how='left')

# 3 continued) Aggregate per (cmdb_id, trace_name)
def agg_trace_group(df):
    count_in_window = int(df.shape[0])
    # If global_P95 is NaN, treat as None and then no points can be > NaN; handle safely
    gp95 = df['global_P95'].iloc[0]
    if pd.isna(gp95):
        gp95_val = None
        above_mask = pd.Series([False]*df.shape[0], index=df.index)
    else:
        gp95_val = float(gp95)
        above_mask = df['value'] > gp95_val
    count_above = int(above_mask.sum())
    earliest_above = df.loc[above_mask, 'ts'].min().isoformat() if count_above > 0 else None
    max_in_window = float(df['value'].max()) if df.shape[0] > 0 else None
    return pd.Series({
        'global_P95': gp95_val,
        'count_points_in_window': count_in_window,
        'count_points_above_P95': count_above,
        'earliest_timestamp_of_point_above_P95': earliest_above,
        'max_value_in_window': max_in_window
    })

if not trace_merged.empty:
    trace_results = (
        trace_merged
        .groupby(['cmdb_id', 'trace_name'], dropna=False)
        .apply(agg_trace_group)
        .reset_index()
    )
else:
    trace_results = pd.DataFrame(columns=['cmdb_id','trace_name','global_P95','count_points_in_window',
                                          'count_points_above_P95','earliest_timestamp_of_point_above_P95',
                                          'max_value_in_window'])

# 4) Sort by count_points_above_P95 desc, then max_value_in_window desc; limit top 20
trace_results_sorted = trace_results.sort_values(
    by=['count_points_above_P95','max_value_in_window'],
    ascending=[False, False]
).head(20).reset_index(drop=True)

# If no row has count_points_above_P95 > 0, prepare explicit note
if trace_results_sorted['count_points_above_P95'].sum() == 0:
    trace_note = "No anomalies found: count_points_above_P95 == 0 for all selected trace series in the window."
else:
    trace_note = None

# Keep output compact: selected columns only
trace_out = trace_results_sorted[['cmdb_id','trace_name','global_P95','count_points_in_window',
                                  'count_points_above_P95','earliest_timestamp_of_point_above_P95',
                                  'max_value_in_window']]

# Display compact results (variable output)
trace_out, trace_note

```
Out[3]:
```
Summary of trace analysis (global P95 computed on full series; window = 2024-01-19 12:00:11 → 2024-01-19 12:30:11 UTC):

- Overall result: Several trace series related to the five candidate services show values above their global P95 inside the incident window. The top 20 series all have 25 points in-window and 2 points above the global P95 each.

- Highest-count / highest-max examples:
  - frontendservice → trace.to_productcatalogservice.row_count: global_P95=3835.0, 25 points in window, 2 points above P95, earliest above = 2024-01-19T12:13:00+00:00, max=3909.0.
  - productcatalogservice → trace.from_frontendservice.row_count: global_P95=3835.0, 25 pts, 2 above, earliest above = 2024-01-19T12:13:00+00:00, max=3909.0.
  - currencyservice → trace.from_frontendservice.row_count and frontendservice → trace.to_currencyservice.row_count: global_P95≈2362.4, each 25 pts, 2 above, earliest above ≈ 2024-01-19T12:07:00+00:00, max≈2415.0.
  - frontendservice → trace.to_recommendationservice.row_count and recommendationservice → trace.from_recommendationservice.row_count: global_P95=589.4, each 25 pts, 2 above, earliest above = 2024-01-19T12:06:00+00:00, max=598.0.
  - checkoutservice-related row_count traces (from/to/ root) also show 2 points above P95, earliest above ≈ 2024-01-19T12:04:00+00:00, max values ~ 314, 158, etc.
  - Additional productcatalogservice, recommendationservice, and paymentservice related trace row_count metrics appear similarly (each with 2 points above P95, earliest ≈ 12:04–12:06, max values shown in table).

- Interpretation: The anomalies are concentrated in row_count-type trace features (requests/rows) between services (frontend ↔ productcatalog, frontend ↔ currency, frontend ↔ recommendation, checkout ↔ others). Each affected series has two in-window samples exceeding its global 95th percentile, with earliest exceedances clustered around ~12:04–12:13 UTC depending on the interaction.

- Actionable note: Focus on the cross-service request/row_count spikes around 12:04–12:13 UTC (frontend → productcatalog/currency/recommendation and checkout interactions). These trace spikes likely indicate increased request load or delayed processing on those paths and are the top trace anomalies in the window.

The original code execution output of IPython Kernel is also provided below for reference:

(                  cmdb_id                                  trace_name  global_P95  count_points_in_window  count_points_above_P95 earliest_timestamp_of_point_above_P95  max_value_in_window
0         frontendservice    trace.to_productcatalogservice.row_count      3835.0                    25.0                     2.0             2024-01-19T12:13:00+00:00               3909.0
1   productcatalogservice        trace.from_frontendservice.row_count      3835.0                    25.0                     2.0             2024-01-19T12:13:00+00:00               3909.0
2         currencyservice        trace.from_frontendservice.row_count      2362.4                    25.0                     2.0             2024-01-19T12:07:00+00:00               2415.0
3         frontendservice          trace.to_currencyservice.row_count      2362.4                    25.0                     2.0             2024-01-19T12:07:00+00:00               2415.0
4         frontendservice    trace.to_recommendationservice.row_count       589.4                    25.0                     2.0             2024-01-19T12:06:00+00:00                598.0
5   productcatalogservice  trace.from_recommendationservice.row_count       589.4                    25.0                     2.0             2024-01-19T12:06:00+00:00                598.0
6   recommendationservice        trace.from_frontendservice.row_count       589.4                    25.0                     2.0             2024-01-19T12:06:00+00:00                598.0
7   recommendationservice  trace.from_recommendationservice.row_count       589.4                    25.0                     2.0             2024-01-19T12:06:00+00:00                598.0
8   recommendationservice    trace.to_productcatalogservice.row_count       589.4                    25.0                     2.0             2024-01-19T12:06:00+00:00                598.0
9   recommendationservice    trace.to_recommendationservice.row_count       589.4                    25.0                     2.0             2024-01-19T12:06:00+00:00                598.0
10        checkoutservice        trace.from_checkoutservice.row_count       307.0                    25.0                     2.0             2024-01-19T12:04:00+00:00                314.0
11        checkoutservice          trace.to_checkoutservice.row_count       307.0                    25.0                     2.0             2024-01-19T12:04:00+00:00                314.0
12        checkoutservice                   trace.from_root.row_count       141.8                    25.0                     2.0             2024-01-19T12:04:00+00:00                158.0
13                   root          trace.to_checkoutservice.row_count       141.8                    25.0                     2.0             2024-01-19T12:04:00+00:00                158.0
14        checkoutservice          trace.to_currencyservice.row_count       118.6                    25.0                     2.0             2024-01-19T12:04:00+00:00                134.0
15        currencyservice        trace.from_checkoutservice.row_count       118.6                    25.0                     2.0             2024-01-19T12:04:00+00:00                134.0
16        checkoutservice    trace.to_productcatalogservice.row_count        83.6                    25.0                     2.0             2024-01-19T12:04:00+00:00                 98.0
17  productcatalogservice        trace.from_checkoutservice.row_count        83.6                    25.0                     2.0             2024-01-19T12:04:00+00:00                 98.0
18        checkoutservice           trace.to_paymentservice.row_count        35.8                    25.0                     2.0             2024-01-19T12:04:00+00:00                 39.0
19         paymentservice        trace.from_checkoutservice.row_count        35.8                    25.0                     2.0             2024-01-19T12:04:00+00:00                 39.0, None)```
```